In [16]:
from funcs import get_json, save_to_json

In [24]:
posts = get_json("posts_2")
raw_user_data = get_json("userdata_2")

In [25]:
def print_basic_stats(users, posts_dict):
    # -------------------------
    # Authors followed per user
    # -------------------------
    follow_counts = [
        len(u.get("follows_authors", []))
        for u in users.values()
    ]

    followees_count = [u.get("stats").get("follows") for u in users.values()]
    avg = sum(followees_count)/len(followees_count)

    avg_authors_followed = (
        sum(follow_counts) / len(follow_counts)
        if follow_counts else 0
    )
    max_authors_followed = max(follow_counts) if follow_counts else 0
    # -------------------------
    # Reposts per post
    # -------------------------
    repost_counts = [
        len(p.get("reposted_by", []))
        for p in posts_dict.values()
        if "reposted_by" in p
    ]
    pos_instances = sum(repost_counts)

    avg_reposts_per_post = (
        sum(repost_counts) / len(repost_counts)
        if repost_counts else 0
    )
    max_reposts_per_post = max(repost_counts) if repost_counts else 0

    posts_with_reposts = sum(c > 0 for c in repost_counts)
    pct_posts_with_reposts = (
        100 * posts_with_reposts / len(posts_dict)
        if repost_counts else 0
    )

    # -------------------------
    # Reposts per user (from reposter_dict output)
    # -------------------------
    reposts_per_user = [
        len(u.get("reposted_posts", []))
        for u in users.values()
    ]
    pos_instances_2 = sum(reposts_per_user)

    users_with_reposts = sum(c > 0 for c in reposts_per_user)
    pct_users_with_reposts = (
        100 * users_with_reposts / len(reposts_per_user)
        if reposts_per_user else 0
    )

    # -------------------------
    # Repost timestamps (from history)
    # -------------------------
    total_reposts = 0
    reposts_with_time = 0

    for u in users.values():
        for h in u.get("history", []):
            if h.get("activity_type") == "repost":
                total_reposts += 1
                if h.get("reposted_at"):
                    reposts_with_time += 1

    pct_reposts_with_time = (
        100 * reposts_with_time / total_reposts
        if total_reposts else 0
    )

    # -------------------------
    # Print results
    # -------------------------
    print("===== BASIC DATASET STATS =====")
    print(f"Users: {len(users)}")
    print(f"Posts: {len(posts_dict)}")
    print()

    print(
        f"Authors followed per user → "
        f"avg: {avg_authors_followed:.2f}, "
        f"max: {max_authors_followed}"
    )
    print(
        f"Avg #followees per user → "
        f"avg: {avg:.2f}, "

    )

    print(
        f"Reposts per post → "
        f"avg: {avg_reposts_per_post:.2f}, "
        f"max: {max_reposts_per_post}"
    )

    print(
        f"Posts with ≥1 repost → "
        f"{posts_with_reposts} "
        f"({pct_posts_with_reposts:.2f}%)"
    )

    print(
        f"Users with ≥1 repost → "
        f"{users_with_reposts} "
        f"({pct_users_with_reposts:.2f}%)"
    )

    print(
        f"Reposts with timestamp → "
        f"{reposts_with_time}/{total_reposts} "
        f"({pct_reposts_with_time:.2f}%)"
    )

    print(
        f"Positive instances: "
        f"{pos_instances} (from posts) and "
        f"{pos_instances_2} (from users)"
        )


In [29]:
def print_posts_with_more_than_x_reposts(posts_dict,x):
    count = sum(
        p.get("repostCount", 0) > x
        for p in posts_dict.values()
    )
    print(f"Posts with >{x} reposts: {count}/{len(posts_dict)}")


print_posts_with_more_than_x_reposts(posts,0)

Posts with >0 reposts: 61307/207450


In [30]:
print_basic_stats(raw_user_data,posts)

===== BASIC DATASET STATS =====
Users: 90687
Posts: 207450

Authors followed per user → avg: 56.47, max: 6442
Avg #followees per user → avg: 1604.19, 
Reposts per post → avg: 1.63, max: 100
Posts with ≥1 repost → 60814 (29.32%)
Users with ≥1 repost → 51002 (56.24%)
Reposts with timestamp → 2498235/2498235 (100.00%)
Positive instances: 337967 (from posts) and 140701 (from users)


In [ ]:
#Ziming had 44,000 pos instances. hence 1:5 dataset was 44,000 x 6

In [36]:
def count_unique_authors(posts_dict):
    """
    Returns the number of unique author DIDs in posts_dict.
    """
    return len({
        (
            p.get("author", {}).get("did")
            or p.get("post", {}).get("author", {}).get("did")
        )
        for p in posts_dict.values()
        if p
    })
num_authors = count_unique_authors(posts)
print(f"Unique authors: {num_authors}")

Unique authors: 48558


In [38]:
def count_reposted_posts_per_hashtag(posts_dict):
    counts = {}

    for post in posts_dict.values():
        source = post.get("_source")
        reposters = post.get("reposted_by")

        if source and len(reposters) > 0:
            counts[source] = counts.get(source, 0) + 1

    print("------- Pos instances -------")
    for source, count in sorted(counts.items()):
        print(f"{source}: {count}")

count_reposted_posts_per_hashtag(posts)

------- Pos instances -------
AI: 4803
Minneapolis: 6028
Olympics: 3258
Pokemon: 8109
TheTraitors: 2425
Trump: 5187
aew: 4725
art: 9965
booksky: 7600
gaza: 4297
superbowl: 4417
